Recursion is a function calling itself. That one sentence sounds simple, but the consequences are deep: it is the natural language for trees, graphs, divide-and-conquer algorithms, and dynamic programming. Before you can truly understand any of those topics, you need to understand recursion — how it maps to the call stack, where it goes wrong, and how to make it efficient. This notebook builds that foundation.

## The Two Essential Parts

Every correct recursive function has exactly two parts:

1. **Base case** — the condition under which the function stops calling itself and returns a direct answer. Without this, the function recurses forever until the call stack overflows.
2. **Recursive case** — the step where the function calls itself with a *smaller* or *simpler* version of the problem, moving toward the base case.

The key insight: each recursive call must make progress toward the base case. If it does not, you have infinite recursion.

### Python

In [ ]:
def factorial(n: int) -> int:
    # Base case — stop recursing
    if n <= 1:
        return 1
    # Recursive case — smaller problem (n-1 instead of n)
    return n * factorial(n - 1)

print(factorial(5))   # 120
print(factorial(0))   # 1
print(factorial(10))  # 3628800

### Kotlin

```kotlin
fun factorial(n: Int): Long {
    if (n <= 1) return 1L        // base case
    return n * factorial(n - 1)  // recursive case
}

fun main() {
    println(factorial(5))   // 120
    println(factorial(10))  // 3628800
}
```

## The Call Stack

Every function call — recursive or not — pushes a **stack frame** onto the call stack. The frame stores the function's arguments, local variables, and the return address (where to resume execution after the function returns).

When a function returns, its frame is popped off the stack and control returns to the caller.

For `factorial(4)`, the stack grows four frames deep before hitting the base case, then unwinds — each frame multiplies its `n` by the result returned from below.

![Call Stack Frames](https://raw.githubusercontent.com/schemabotview/dsa/main/img/call-stack-frames.svg)

## Visualising the Call Stack in Python

### Python

In [ ]:
import sys

def factorial_traced(n: int, depth: int = 0) -> int:
    indent = "  " * depth
    print(f"{indent}→ factorial({n})")
    if n <= 1:
        print(f"{indent}← base case: return 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}← return {n} × ... = {result}")
    return result

factorial_traced(4)

## Recursion Tree — Fibonacci

The Fibonacci sequence is defined recursively: `fib(n) = fib(n-1) + fib(n-2)`, with base cases `fib(0) = 0` and `fib(1) = 1`.

The naive recursive implementation looks clean but hides a severe problem: it recomputes the same subproblems many times. `fib(3)` is computed twice, `fib(2)` three times. For `fib(50)` the call tree has trillions of nodes.

![Recursion Tree — Fibonacci](https://raw.githubusercontent.com/schemabotview/dsa/main/img/recursion-tree-fibonacci.svg)

### Python

In [ ]:
# Naive — O(2^n) time. Do not use for large n.
def fib_naive(n: int) -> int:
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

call_count = 0
def fib_counted(n: int) -> int:
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib_counted(n - 1) + fib_counted(n - 2)

for n in [5, 10, 20, 30]:
    call_count = 0
    result = fib_counted(n)
    print(f"fib({n:2d}) = {result:8d}   calls = {call_count:,}")

## Memoization — Cache Recursive Results

**Memoization** is the technique of caching the result of each unique subproblem the first time it is computed, and returning the cached result on any subsequent call. This converts the exponential recursion tree into a linear one — each unique input is computed exactly once.

Memoization is the bridge between naive recursion and dynamic programming. DP notebooks 20 and 21 build directly on this concept.

### Python

In [ ]:
from functools import lru_cache
import time

# Manual memoization
def fib_memo(n: int, cache: dict = {}) -> int:
    if n in cache:
        return cache[n]          # return cached result immediately
    if n <= 1:
        return n
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]

# Using Python's built-in LRU cache decorator
@lru_cache(maxsize=None)
def fib_cached(n: int) -> int:
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

# Compare performance
t0 = time.perf_counter()
print(fib_naive(35))
print(f"naive :  {time.perf_counter() - t0:.3f}s")

t0 = time.perf_counter()
print(fib_cached(35))
print(f"cached:  {time.perf_counter() - t0:.6f}s")

print(fib_cached(1000))   # instant — no stack overflow either with iterative DP

### Kotlin

```kotlin
// Manual memoization with a map
val cache = mutableMapOf<Int, Long>()

fun fibMemo(n: Int): Long {
    if (n <= 1) return n.toLong()
    cache[n]?.let { return it }     // return cached result if present
    val result = fibMemo(n - 1) + fibMemo(n - 2)
    cache[n] = result
    return result
}
```

## Stack Overflow — Recursion Depth Limits

The call stack has a fixed maximum size. In Python the default recursion limit is 1,000 frames. Exceeding it raises a `RecursionError`. In Kotlin (JVM), the stack is typically 512 KB–1 MB — deep recursion will throw a `StackOverflowError`.

For algorithms on large inputs, this is a real constraint. The solutions are:
1. **Convert to iteration** — use an explicit stack (a `list` or `deque`) instead of the call stack
2. **Increase the limit** (Python: `sys.setrecursionlimit`) — a workaround, not a fix
3. **Use tail recursion** (not optimised in Python or JVM Kotlin — see note below)

### Python

In [ ]:
import sys
print(sys.getrecursionlimit())   # 1000 by default

# Converting factorial to iteration — no stack depth risk
def factorial_iter(n: int) -> int:
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(factorial_iter(1000))   # works — no RecursionError

## Common Recursion Patterns

### Divide and Conquer — Binary Search

### Python

In [ ]:
# Recursive binary search — O(log n) time, O(log n) space (call stack depth)
def binary_search(arr: list[int], target: int, left: int, right: int) -> int:
    if left > right:                  # base case — not found
        return -1
    mid = (left + right) // 2
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search(arr, target, mid + 1, right)   # search right half
    else:
        return binary_search(arr, target, left, mid - 1)    # search left half

arr = [1, 3, 5, 7, 9, 11, 13, 15]
print(binary_search(arr, 7,  0, len(arr) - 1))   # 3
print(binary_search(arr, 10, 0, len(arr) - 1))   # -1

### Kotlin

```kotlin
fun binarySearch(arr: List<Int>, target: Int, left: Int, right: Int): Int {
    if (left > right) return -1
    val mid = (left + right) / 2
    return when {
        arr[mid] == target -> mid
        arr[mid] < target  -> binarySearch(arr, target, mid + 1, right)
        else               -> binarySearch(arr, target, left, mid - 1)
    }
}
```

### Structural Recursion — Flatten a Nested List

### Python

In [ ]:
# Recursion is natural when the data structure is itself recursive
def flatten(nested: list) -> list:
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))   # recurse into sub-list
        else:
            result.append(item)
    return result

print(flatten([1, [2, [3, 4]], [5, 6], 7]))
# [1, 2, 3, 4, 5, 6, 7]

### Kotlin

```kotlin
fun flatten(nested: List<Any>): List<Int> {
    val result = mutableListOf<Int>()
    for (item in nested) {
        when (item) {
            is List<*> -> result.addAll(flatten(item as List<Any>))
            is Int     -> result.add(item)
        }
    }
    return result
}
```

### Power Function — Fast Exponentiation

### Python

In [ ]:
# Naive: O(n) multiplications
# Fast exponentiation: O(log n) — halve the exponent at each step
def power(base: float, exp: int) -> float:
    if exp == 0:           # base case
        return 1
    if exp % 2 == 0:       # even exponent: x^n = (x^(n/2))^2
        half = power(base, exp // 2)
        return half * half
    else:                  # odd exponent: x^n = x * x^(n-1)
        return base * power(base, exp - 1)

print(power(2, 10))    # 1024.0
print(power(3, 5))     # 243.0
print(power(2, 32))    # 4294967296.0  — computed in 6 steps not 32

### Kotlin

```kotlin
fun power(base: Double, exp: Int): Double {
    if (exp == 0) return 1.0
    if (exp % 2 == 0) {
        val half = power(base, exp / 2)
        return half * half
    }
    return base * power(base, exp - 1)
}
```

## Recursion vs Iteration

| | Recursion | Iteration |
|---|---|---|
| Code clarity | Often cleaner for tree/graph/divide-and-conquer | Cleaner for simple loops |
| Stack usage | O(depth) call stack frames | O(1) unless explicit stack used |
| Stack overflow risk | Yes — deep recursion hits limit | No |
| Performance | Slight overhead per call | Generally faster |
| Tail-call optimisation | Not in Python or JVM | N/A |

**Rule of thumb:** Use recursion when the problem is naturally recursive (trees, graphs, divide-and-conquer). Convert to iteration when depth is unbounded or performance is critical. You can always simulate recursion with an explicit stack.

### Python — Converting Recursion to Iteration with an Explicit Stack

In [ ]:
# Iterative factorial using an explicit stack — avoids RecursionError
def factorial_stack(n: int) -> int:
    stack = list(range(2, n + 1))   # [2, 3, 4, ..., n]
    result = 1
    while stack:
        result *= stack.pop()
    return result

print(factorial_stack(10))    # 3628800
print(factorial_stack(1000))  # exact — no recursion depth issue

## Key Takeaways

- Every recursive function needs a **base case** (stop) and a **recursive case** (smaller problem). Missing the base case → infinite recursion → stack overflow.
- Each recursive call pushes a **stack frame** onto the call stack. Frames are popped in LIFO order as functions return — the call stack is literally a stack.
- **Memoization** caches the result of each unique subproblem, converting exponential recursion trees into linear work. Use `@lru_cache` in Python or a `Map` in Kotlin.
- Python's default recursion limit is 1,000 frames. For deep recursion, convert to an explicit stack.
- **Divide and conquer** (binary search, merge sort) and **structural recursion** (trees, nested data) are the two most natural recursion shapes.
- **Fast exponentiation** halves the exponent at each step — O(log n) multiplications instead of O(n).
- You can always replace recursion with an explicit stack + a loop. The call stack *is* a stack — making it explicit just moves the structure from implicit to visible.